# EASY-IMAGE-GENERATOR
`EASY-IMAGE-GENERATOR` is a project that allows anyone to easily generate images using Google Colab.

[GitHub](https://github.com/YooJH0/EASY-IMAGE-GENERATOR/)

In [ ]:
# INSTALL LIBRARY
!pip install -q \
  accelerate==1.1.1 \
  datasets==3.1.0 \
  diffusers==0.32.2 \
  peft==0.13.2 \
  transformers==4.46.3 \
  safetensors

# IMPORT LIBRARY
import json
import os
from pathlib import Path
import subprocess
import sys
import urllib.request
from diffusers import StableDiffusionPipeline
from google.colab import drive
from PIL import Image
import torch

# DECLARE CONSTANT
BASE_DIR = "/content/drive/MyDrive/EASY-IMAGE-GENERATOR/"
DATASET_DIR = BASE_DIR + "dataset/"
IMAGE_DIR = BASE_DIR + "image/"
MODEL_DIR = BASE_DIR + "model/"
SKRIPT_DIR = BASE_DIR + "script/"
TEXT_DIR = BASE_DIR + "text/"
SKRIPT_FILE = SKRIPT_DIR + "train_text_to_image_lora.py"

# DECLARE FUNCTION - MAKE DIRECTORYS
def mkdir(dir):
  os.makedirs(dir, exist_ok = True)

# DECLARE FUNCTION - VALIDATE TEXT
def validate_text(my_text):
  texts = {}

  if not os.path.isfile(my_text):
    raise FileNotFoundError(f"MY TEXT FILE NOT FOUND: {my_text}")

  with open(my_text, "r", encoding = "utf-8-sig") as file:
    for line_number, line in enumerate(file, 1):
      line = line.strip()

      if not line:
        continue

      file_name, seperator, text = line.partition(":")
      file_name = file_name.strip()
      text = text.strip()

      if not file_name or not seperator or not text:
        raise ValueError(f"MY TEXT INVALID: {line_number}")

      if file_name in texts:
        raise ValueError(f"MY TEXT INVALID: {file_name}")

      texts[file_name] = text

  return texts

# DECLARE FUNCTION - VALIDATE DATASET
def validate_dataset(texts, my_dataset):
  records = []

  if not os.path.isdir(my_dataset):
    raise FileNotFoundError(f"MY DATASET FILE NOT FOUND: {my_dataset}")

  for file_name, text in texts.items():
    my_image = os.path.join(my_dataset, file_name)

    if not os.path.isfile(my_image):
      raise FileNotFoundError(f"MY IMAGE FILE NOT FOUND: {my_image}")

    if Path(my_image).suffix.lower() not in {".jpeg", ".jpg", ".png", ".webp"}:
      raise ValueError(f"MY IMAGE INVALID: {my_image}")

    with Image.open(my_image) as image:
      image.verify()

    records.append({
      "file_name": file_name,
      "text": text
    })

  if not records:
    raise ValueError("MY IMAGE INVALID")

  return records

# DECLARE FUNCTION - CREATE METADATA
def create_metadata(my_dataset, records):
  my_metadata = os.path.join(my_dataset, "metadata.jsonl")

  with open(my_metadata, "w", encoding = "utf-8") as file:
    for record in records:
      file.write(json.dumps(record, ensure_ascii = False) + "\n")

  return my_metadata

# DECLARE FUNCTION - TRAIN MODEL
def train_model(my_text, my_dataset, my_model):
  my_text = os.path.join(TEXT_DIR, my_text)
  my_dataset = os.path.join(DATASET_DIR, my_dataset)
  my_model = os.path.join(MODEL_DIR, my_model)
  mkdir(my_model)

  texts = validate_text(my_text)
  records = validate_dataset(texts, my_dataset)
  my_metadata = create_metadata(my_dataset, records)

  command = [
    sys.executable, SKRIPT_FILE,
    "--train_data_dir", my_dataset,
    "--output_dir", my_model,
    "--pretrained_model_name_or_path", "Lykon/dreamshaper-8",
    "--gradient_accumulation_steps", "4",
    "--learning_rate", "1e-4",
    "--max_train_steps", "100",
    "--rank", "4",
    "--resolution", "512",
    "--seed", "0",
    "--train_batch_size", "1",
    "--caption_column", "text",
    "--checkpointing_steps", "200",
    "--dataloader_num_workers", "0",
    "--gradient_checkpointing",
    "--lr_scheduler", "constant",
    "--lr_warmup_steps", "0",
    "--mixed_precision", "fp16",
  ]

  subprocess.run(command, check = True)

  my_model = os.path.join(my_model, "pytorch_lora_weights.safetensors")

  if not os.path.isfile(my_model):
    raise FileNotFoundError("MY MODEL FILE NOT FOUND")

  print(f"MODEL TRAINED: {my_model}")

# DECLARE FUNCTION - GENERATE IMAGE
def generate_image(prompt, my_model, my_image):
  my_model = os.path.join(MODEL_DIR, my_model)
  my_image = os.path.join(IMAGE_DIR, my_image)
  my_weight = os.path.join(my_model, "pytorch_lora_weights.safetensors")

  if not os.path.isfile(my_weight):
    raise FileNotFoundError(f"MY MODEL FILE NOT FOUND: {my_weight}")

  pipe = StableDiffusionPipeline.from_pretrained(
    "Lykon/dreamshaper-8",
    safety_checker = None,
    torch_dtype = torch.float16).to("cuda")

  pipe.load_lora_weights(my_model)

  try:
    with torch.inference_mode():
      image = pipe(
        prompt = prompt,
        guidance_scale = 7.5,
        num_inference_steps = 50,
        progress_bar = None).images[0]

    image.save(my_image)

  finally:
    del pipe
    torch.cuda.empty_cache()

  print(f"IMAGE GENERATED: {my_image}")
  display(image)

# MOUNT DRIVE
drive.mount('/content/drive')

# MAKE DIRECTORYS
mkdir(BASE_DIR)
mkdir(DATASET_DIR)
mkdir(IMAGE_DIR)
mkdir(MODEL_DIR)
mkdir(SKRIPT_DIR)
mkdir(TEXT_DIR)

# DOWNLOAD FILE
if not os.path.isfile(SKRIPT_FILE):
  urllib.request.urlretrieve("https://raw.githubusercontent.com/huggingface/diffusers/v0.32.2/examples/text_to_image/train_text_to_image_lora.py", SKRIPT_FILE)

In [ ]:
# TRAIN MODEL
MY_TEXT = "my_text.txt"
MY_DATASET = "my_dataset/"
MY_MODEL = "my_model/"

train_model(MY_TEXT, MY_DATASET, MY_MODEL)

In [ ]:
# GENERATE IMAGE
PROMPT = "20 years old, male, student"
MY_MODEL = "my_model/"
MY_IMAGE = "my_image.png"

generate_image(PROMPT, MY_MODEL, MY_IMAGE)